In [1]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 1. Load Data
df = pd.read_csv(r"D:\SMIT income predictor project\data\adult.csv")

# Clean target variable
df['income'] = df['income'].str.strip().str.replace('.', '', regex=False)
X = df.drop(columns=['income'])
y = df['income'].apply(lambda x: 1 if '>50K' in x else 0)

# --- OPTIMIZATION STEP: DOWNSAMPLE FOR FASTER TRAINING ---
# Take a random 15,000 row subset so SVC and KNN run in seconds instead of hours
df_sampled = df.sample(n=15000, random_state=42)
X_sampled = df_sampled.drop(columns=['income'])
y_sampled = df_sampled['income'].apply(lambda x: 1 if '>50K' in x else 0)

X_train, X_test, y_train, y_test = train_test_split(X_sampled, y_sampled, test_size=0.2, random_state=42)

# 2. Identify Column Types
numerical_cols = ['age', 'fnlwgt', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_cols = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country']

X_train = X_train[numerical_cols + categorical_cols]
X_test = X_test[numerical_cols + categorical_cols]

# 3. Create Preprocessing Bundles
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# 4. Train and Compare Models (With speed tweaks)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    # Added max_iter=2000 so SVC stops calculating once it hits a reasonable threshold
    'SVC': SVC(probability=True, max_iter=2000, random_state=42), 
    # n_jobs=-1 tells KNN to use all your computer's CPU cores to run faster
    'KNN': KNeighborsClassifier(n_jobs=-1) 
}

best_accuracy = 0
best_model = None

for name, model in models.items():
    print(f"Training {name}...") # This helps you track progress live
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipeline.fit(X_train, y_train)
    acc = accuracy_score(y_test, pipeline.predict(X_test))
    print(f"{name} Accuracy: {acc:.4f}\n")
    
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = pipeline

# 5. Export Best Pipeline using pickle
with open('model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print("Saved best model to model.pkl successfully!")


Training Logistic Regression...
Logistic Regression Accuracy: 0.8707

Training Decision Tree...
Decision Tree Accuracy: 0.8127

Training SVC...


c:\Users\MYC\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


SVC Accuracy: 0.8707

Training KNN...
KNN Accuracy: 0.8490

Saved best model to model.pkl successfully!


In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\MYC\Downloads\adult.csv")
df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
